# 🤖 Agente de Código Full Stack

Agente que recibe instrucciones en lenguaje natural y genera una **app web completa en Next.js**.

**Stack:**
- **GitHub Models** (`gpt-4o`) como LLM
- **E2B** como sandbox seguro para escribir y compilar código
- **Runtime Summary** para no quedarse sin contexto en tareas largas

**Credenciales necesarias (Colab Secrets 🔑):**
- `GITHUB_TOKEN` → https://github.com/settings/tokens
- `E2B_API_KEY`  → https://e2b.dev


## 1. Instalación

In [ ]:
!pip install openai e2b-code-interpreter -q


## 2. Credenciales

In [ ]:
import os
from google.colab import userdata

os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
os.environ['E2B_API_KEY']  = userdata.get('E2B_API_KEY')

print('✅ Credenciales cargadas')


## 3. Clonar / montar el repositorio

Si estás en Colab, clona el repo. Si ya tenés el código en el entorno, saltá esta celda.


In [ ]:
# Opción A: clonar desde GitHub
!git clone https://github.com/mijaelryan/fullstack-agent.git
%cd fullstack-agent

# Opción B: ya estás en el directorio correcto
# import os
# print('Directorio actual:', os.getcwd())


## 4. Inicializar cliente y sandbox

In [ ]:
from agent import build_client, build_sandbox

client = build_client()   # usa os.environ['GITHUB_TOKEN']
sbx    = build_sandbox()  # usa os.environ['E2B_API_KEY']


## 5. Verificación rápida de herramientas (Parte 1)

Antes de correr el agente, comprobamos que las herramientas del sandbox funcionan.


In [ ]:
from lib.sbx_tools import list_directory, write_file, read_file, search_file_content

# Escribir un archivo de prueba
r = write_file(sbx, '/home/user/test.txt', 'Hola desde sbx_tools!\nLínea 2')
print('write_file:', r)

# Leerlo
r = read_file(sbx, '/home/user/test.txt')
print('read_file:', r)

# Listar directorio
r = list_directory(sbx, '/home/user')
print('list_directory:', r)

# Buscar patrón
r = search_file_content(sbx, 'Hola', path='/home/user')
print('search_file_content:', r)


## 6. Verificación de compresión de contexto

Simulamos un historial que supere el límite de `4,000` tokens estimados
para verificar que `maybe_compress` actúa correctamente.
El historial de prueba se mantiene por debajo de `8,000` tokens
para no superar el límite del modelo de resumen (`gpt-4o-mini`).


In [ ]:
from lib.context import count_tokens, maybe_compress, MAX_TOKENS

# Crear historial falso que supere el límite de 4k tokens
# pero no el límite del modelo de resumen (8k).
# Cada par user/assistant ocupa ~250 tokens → 20 pares = ~5,000 tokens.
fake_messages = [
    {"role": "user",      "content": [{"text": "x" * 500}]},
    {"role": "assistant", "content": [{"text": "y" * 500}]},
] * 20  # ~5,000 tokens estimados — supera 4k pero no 8k

tokens_antes = count_tokens(fake_messages)
print(f'Tokens estimados: {tokens_antes:,}  |  Límite: {MAX_TOKENS:,}')
assert tokens_antes > MAX_TOKENS, 'El historial debe superar el límite'

compressed = maybe_compress(fake_messages, client)
print(f'Mensajes después de comprimir: {len(compressed)} (era {len(fake_messages)})')
assert len(compressed) < len(fake_messages), 'La compresión debe reducir los mensajes'
print('✅ Compresión de contexto OK')


## 7. Probar el agente

### Tarea 1: Crear la app


In [ ]:
from agent import run_agent

historial = []

historial, respuesta = run_agent(
    'Crea una app de lista de tareas con título, input para agregar tareas y botón Agregar.',
    client=client,
    sbx=sbx,
    messages=historial,
)

print('\n--- RESPUESTA FINAL ---')
print(respuesta)


### Tarea 2: Ajuste sobre el mismo historial

El agente recuerda que existe una lista de tareas sin que se lo repitamos.
Pedimos una mejora que solo tiene sentido si la Tarea 1 ya fue ejecutada.


In [ ]:
# El agente recuerda el contexto: sabe que hay tareas, que tienen un componente,
# y puede editarlo sin que se le describa la estructura del proyecto.
historial, respuesta = run_agent(
    'Agregá un botón Eliminar en cada tarea y mostrá un contador de tareas pendientes arriba.',
    client=client,
    sbx=sbx,
    messages=historial,
)

print('\n--- RESPUESTA FINAL ---')
print(respuesta)


## 8. Verificación de upload_file (nueva en v2.0)

Sube un archivo binario desde el entorno local al sandbox usando `upload_file`.
Verifica que los bytes llegaron intactos comprobando el tamaño en el sandbox.


In [ ]:
import struct, zlib, os
from lib.sbx_tools import upload_file

def _make_minimal_png(path):
    """Genera un PNG 1x1 px válido para usar como asset de prueba."""
    def chunk(name, data):
        c = zlib.crc32(name + data) & 0xFFFFFFFF
        return struct.pack('>I', len(data)) + name + data + struct.pack('>I', c)
    png  = b'\x89PNG\r\n\x1a\n'
    png += chunk(b'IHDR', struct.pack('>IIBBBBB', 1, 1, 8, 2, 0, 0, 0))
    png += chunk(b'IDAT', zlib.compress(b'\x00\xff\xff\xff'))
    png += chunk(b'IEND', b'')
    with open(path, 'wb') as f:
        f.write(png)
    return len(png)

local_img = 'test_sample.png'
sbx_img   = '/home/user/test_sample.png'

local_size = _make_minimal_png(local_img)
print(f'[local] Imagen creada: {local_img} ({local_size} bytes)')

r = upload_file(sbx, local_path=local_img, sbx_path=sbx_img)
print(f'[sandbox] {r["message"]}')

# Verificar tamaño real en sandbox (no usamos read_file — lee en modo texto)
exe  = sbx.run_code(f"import os; print(os.path.getsize('{sbx_img}'))")
size = int(''.join(exe.logs.stdout).strip() or '0')
ok   = size == local_size
print(f'[sandbox] Tamaño recibido: {size} bytes | esperado: {local_size} bytes → {"✅ OK" if ok else "❌ FALLÓ"}')

os.remove(local_img)
print('[local] Archivo temporal eliminado')


## 9. Cerrar el sandbox


In [ ]:
sbx.kill()
print('✅ Sandbox cerrado')


---
## Resumen de arquitectura

| Componente | Archivo | Rol |
|---|---|---|
| Herramientas filesystem | `lib/sbx_tools.py` | list, read, write, search, replace, glob, execute_bash, upload_file |
| Compresión de contexto | `lib/context.py` | Runtime Summary a 4k tokens |
| System prompt | `lib/prompts.py` | Instrucciones Next.js al agente |
| Loop del agente | `agent.py` | `run_agent()` con tool calls |
| Demo | `notebook.ipynb` | Este archivo |

**Flujo:**
```
run_agent(query)
  → maybe_compress()   # comprime si >4k tokens estimados
  → llm()              # GitHub Models gpt-4o
  → execute_tool()     # sbx_tools en E2B
  → _try_download()    # descarga workspace/ al sistema local
  → loop hasta sin tool calls
```

> **Nota técnica:** El agente usa `gpt-4o` para generar código y `gpt-4o-mini`
> para comprimir el historial — optimización intencional de tokens.
